In [ ]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
from ta.momentum import RSIIndicator
import time

# Initialize MT5 connection
def initialize_mt5():
    if not mt5.initialize():
        print("MT5 initialization failed")
        return False
    print("MT5 initialized successfully")
    return True

# Get account info
def get_account_info():
    account_info = mt5.account_info()
    if account_info is None:
        print("Failed to get account info")
        return None
    return account_info

# Check if there is an open position
def check_open_position(symbol):
    positions = mt5.positions_get(symbol=symbol)
    if positions is None:
        print("Failed to get positions")
        return None, None
    buy_position = False
    sell_position = False
    for pos in positions:
        if pos.type == mt5.POSITION_TYPE_BUY:
            buy_position = True
        elif pos.type == mt5.POSITION_TYPE_SELL:
            sell_position = True
    return buy_position, sell_position

# Get price data
def get_price_data(symbol, timeframe, num_bars):
    rates = mt5.copy_rates_from_pos(symbol, timeframe, 0, num_bars)
    if rates is None:
        print("Failed to get price data")
        return None
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    return df

# Calculate RSI
def calculate_rsi(df, period=14):
    rsi = RSIIndicator(df['close'], period)
    return rsi.rsi()

# Place order
def place_order(symbol, order_type, volume, price, sl, tp):
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": symbol,
        "volume": volume,
        "type": order_type,
        "price": price,
        "sl": sl,
        "tp": tp,
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }
    result = mt5.order_send(request)
    if result.retcode != mt5.TRADE_RETCODE_DONE:
        print(f"Order failed: {result.comment}")
        return False
    print(f"Order placed: {result}")
    return True

# Main trading logic
def trading_logic(symbol, timeframe, upper_threshold, lower_threshold):
    # Calculate range, TP, and SL
    range_value = upper_threshold - lower_threshold
    tp_range = 0.8 * range_value
    sl_range = 0.4 * range_value
    volume = 0.01

    while True:
        # Get price data
        df = get_price_data(symbol, timeframe, 100)
        if df is None:
            time.sleep(60)
            continue

        # Calculate RSI
        rsi = calculate_rsi(df)
        current_rsi = rsi.iloc[-1]
        current_price = df['close'].iloc[-1]

        # Check open positions
        buy_position, sell_position = check_open_position(symbol)
        if buy_position is None or sell_position is None:
            time.sleep(60)
            continue

        # Trading conditions
        # Buy conditions
        if not buy_position:
            # RSI < 20 and price within lower_threshold ± 10
            if current_rsi < 20 and (lower_threshold - 10 <= current_price <= lower_threshold + 10):
                tp = current_price + tp_range
                sl = current_price - sl_range
                print(f"Placing BUY order: Price={current_price}, TP={tp}, SL={sl}")
                place_order(symbol, mt5.ORDER_TYPE_BUY, volume, current_price, sl, tp)
            # RSI < 20 and price above upper_threshold + 10
            elif current_rsi < 20 and current_price > upper_threshold + 10:
                tp = current_price + tp_range
                sl = current_price - sl_range
                print(f"Placing BUY order: Price={current_price}, TP={tp}, SL={sl}")
                place_order(symbol, mt5.ORDER_TYPE_BUY, volume, current_price, sl, tp)

        # Sell conditions
        if not sell_position:
            # RSI > 80 and price below lower_threshold - 10
            if current_rsi > 80 and current_price < lower_threshold - 10:
                tp = current_price - tp_range
                sl = current_price + sl_range
                print(f"Placing SELL order: Price={current_price}, TP={tp}, SL={sl}")
                place_order(symbol, mt5.ORDER_TYPE_SELL, volume, current_price, sl, tp)
            # RSI > 80 and price within upper_threshold ± 10
            elif current_rsi > 80 and (upper_threshold - 10 <= current_price <= upper_threshold + 10):
                tp = current_price - tp_range
                sl = current_price + sl_range
                print(f"Placing SELL order: Price={current_price}, TP={tp}, SL={sl}")
                place_order(symbol, mt5.ORDER_TYPE_SELL, volume, current_price, sl, tp)

        # Wait before next iteration
        time.sleep(60)

# Example usage
if __name__ == "__main__":
    symbol = "EURUSD"
    timeframe = mt5.TIMEFRAME_M15
    upper_threshold = 1.2000  # Set your upper threshold
    lower_threshold = 1.1800  # Set your lower threshold

    if initialize_mt5():
        account_info = get_account_info()
        if account_info:
            print(f"Account: {account_info.login}, Balance: {account_info.balance}")
            trading_logic(symbol, timeframe, upper_threshold, lower_threshold)
        mt5.shutdown()